In [1]:
!pip install -U \
  langchain-core \
  langchain-community \
  langchain-chroma \
  chromadb \
  sentence-transformers \
  openai \
  tiktoken


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 4.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.14.0
    Uninstalling openai-2.14.0:
      Successfully uninstalled openai-2.14.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Uninstalling langchain-core-1.2.6:
      Successfully uninstalled langchain-core-1.2.6


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
from sentence_transformers import SentenceTransformer

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_chroma import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.chat_models import ChatOpenAI
from langchain_community.llms import Ollama

from chromadb.config import Settings
import chromadb


In [4]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://www.educosys.com/course/genai")

docs = loader.load()
print(docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://www.educosys.com/course/genai', 'title': 'Hands-on Generative AI Course', 'description': 'Learn, Build, Deploy and Apply Generative AI', 'language': 'en'}, page_content="Hands-on Generative AI CourseCoursesBundle CoursesMentorFree ContentTestimonialsFAQLogin Signup Starts on 6th Feb, 2026Hands-on Generative AI CourseLearn, Build, Deploy and Apply Generative AI7 weeks · Fri, Sat and Sun · 9:00 PM - 11:00 PM IST (every class) + Post-class doubt supportLast fully LIVE batch. All future courses will follow a hybrid format!Access all Live BatchesLifetime access of RecordingsAccess Discord CommunityCode availableBuild ProjectsLearn Future-Ready TechEnroll 1Week 1Foundations of Generative AI Introduction to AI Mathematical Foundations for AI Probability, Statistics, and Linear Algebra Basics of Neural Networks Gradient Descent and Optimization Basics Architectures: Feedforward, RNN, and CNN Mini Project - Build a Simple Neural Network Using TensorFlow Mi

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(splits[0])

page_content='Hands-on Generative AI CourseCoursesBundle CoursesMentorFree ContentTestimonialsFAQLogin Signup Starts on 6th Feb, 2026Hands-on Generative AI CourseLearn, Build, Deploy and Apply Generative AI7 weeks · Fri, Sat and Sun · 9:00 PM - 11:00 PM IST (every class) + Post-class doubt supportLast fully LIVE batch. All future courses will follow a hybrid format!Access all Live BatchesLifetime access of RecordingsAccess Discord CommunityCode availableBuild ProjectsLearn Future-Ready TechEnroll 1Week 1Foundations of Generative AI Introduction to AI Mathematical Foundations for AI Probability, Statistics, and Linear Algebra Basics of Neural Networks Gradient Descent and Optimization Basics Architectures: Feedforward, RNN, and CNN Mini Project - Build a Simple Neural Network Using TensorFlow Mini Project - Train an Autoencoder on the MNIST Dataset2Week 2Deep Generative Models Discriminative and Generative models Generative Adversarial Networks (GANs) Variational Autoencoders (VAEs) Pro

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc.page_content for doc in splits]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

print(f"Embeddings shape: {len(embeddings)} x {len(embeddings[0])}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: 20 x 384


In [7]:
client = chromadb.Client(
    Settings(persist_directory="./chroma_db")
)

collection = client.get_or_create_collection(name="genai_course")

ids = [f"chunk_{i}" for i in range(len(texts))]
metadatas = [
    {
        "source": doc.metadata.get("source", "unknown"),
        "chunk_id": i
    }
    for i, doc in enumerate(splits)
]

collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=ids,
    metadatas=metadatas
)

#client.persist()
print("✅ Vector store created and persisted")


✅ Vector store created and persisted


In [8]:
embedding_function = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    collection_name="genai_course",
    persist_directory="./chroma_db",
    embedding_function=embedding_function
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


C:\Users\sujoy.seal\AppData\Local\Temp\ipykernel_15048\1372256462.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(


In [9]:
docs = retriever.invoke("What is generative AI?")

for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:300])


In [10]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert AI tutor.
Answer the question using ONLY the context below.
If the answer is not present, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
"""
)


In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini",api_key=OPENAI_API_KEY)


In [12]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [13]:
query = "Explain generative AI in simple terms"

answer = rag_chain.invoke(query)

print("\n🧠 Answer:\n")
print(answer)



🧠 Answer:

Generative AI is a type of artificial intelligence that can create new content, such as text, images, music, or videos, by learning from existing data. It uses patterns and information from what it has been trained on to come up with original works.
